# L7 artery CFD, final: the campaign's CFD stage (physics v3, Fluent 26.1)

One design, one state, one inflow per run. The lumen comes from the mechanical campaign (Vishesh's `build_lumen` on the FEA result), is meshed with his watertight workflow, solved with the physics v3 setup in the CSV, and the result is written as the row his campaign reads (`cfd_stage.csv`, his columns). `run_cfd_stage.py` runs this notebook for every inflow of a design and state with his stage's command line.

| | Value (from the CSV) | Source |
|---|---|---|
| Lumen | From the FEA result for `DESIGN` / `STATE` (blocked: the lineage's analytic L7 lumen, R0 1.22 mm); straight extensions; axis +z, inlet at z = 0 | `build_lumen` (Vishesh) |
| Domain | Full 360° for blocked, quarter for the stented lumen | Vishesh's pipeline domains |
| Mesh | Watertight, throat-adapted sizes, 4 prism layers, poly-hexcore; min orthogonal quality ≥ 0.1, cells < 512,000 | `cfd_pipeline_v3_vishesh.yaml` |
| Blood | 1050 kg/m³, Carreau (μ₀ 0.056, μ∞ 0.00345 Pa·s, λ 3.313 s, n 0.3568) | Cho & Kensey 1991 |
| Inlet | Steady parabolic, 0.10 m/s (rest) or 0.26 m/s (hyperaemia), one per run | Kaufmann 2005 |
| Outlet / wall | 0 Pa gauge / no-slip | 19_CFD_SETTINGS |
| Flow model | Laminar (k-ω waits on Decision 7) | |
| Numerics | Coupled, least-squares gradients, 2nd-order pressure, 2nd-order upwind | 19_CFD_SETTINGS |
| Convergence | Residuals 1e-5 AND Δp flat within 0.05 Pa over 20 iterations | Pipeline yaml |
| Results | Full Δp; segment Δp over the 25 mm artery; face WSS on the artery wall over the segment, area-weighted, with low-WSS fractions | 21_RESULT_DEFINITIONS |

**How to run:** `python run_cfd_stage.py --design <id> --state <state> --u 0.10,0.26 --out <dir> --pipeline-root <repo>` runs every inflow (fresh kernel and Fluent each). By hand: set cell 0, restart the kernel and run all, once per inflow.

**Drop-in folders.** `dropin/inlet_udf/`: drop the pulsatile inlet UDF (`.c`) and the run becomes one transient run at the waveform's mean velocity (no 0.10 / 0.26 pair), settings in `inlet_udf.csv`. `dropin/tawss_osi/`: drop the TAWSS / OSI UDF (`.c`) as well and the run reports TAWSS, OSI and RRT on the artery wall; results in `dropin/tawss_osi/tawss_osi_results.csv`. With no `.c` in them, the run is the steady parabolic one above.

Not in this version: Harshita's boundary-condition module (`bc_stage`); k-ω (Decision 7).

## 0. Run parameters
The only cell you edit. `run_cfd_stage.py` passes these for each run (the cell is tagged `parameters` for papermill).

In [ ]:
import os
NB_FILE  = globals().get("__vsc_ipynb_file__")              # set by VS Code; Jupyter / papermill start in the notebook folder
WORK_DIR = os.path.dirname(NB_FILE) if NB_FILE else os.getcwd()
os.chdir(WORK_DIR)                                           # CSVs and outputs are relative to this folder
print("working folder:", WORK_DIR)

DESIGN         = "L7"
STATE          = "blocked"                         # blocked | expanded | released
INLET_U        = 0.10                              # m/s, this run
U_LIST         = "0.10,0.26"                       # every inflow of this design and state
LINEAGE        = "physics_v3_L7_camp244b"
PIPELINE_ROOT  = "../23_CFD_PIPELINE_v3"           # Vishesh's pipeline (tools/, StenosisCFD/, FullTwin/); his StentForge repo in the campaign
HELD_ROOT      = ""                                # FEA extracts; blank: <PIPELINE_ROOT>/out/physics_v3/fallback/held
CAMPAIGN_OUT   = "artery_runs_v3/campaign"         # results go to <CAMPAIGN_OUT>/<DESIGN>/<STATE>_struts-no/
LUMEN_STEP     = ""                                # test only: a ready lumen STEP (.stl beside it) instead of build_lumen
DROPIN_DIR     = "dropin"                          # inlet_udf/ and tawss_osi/: a .c file there switches it on
REMESH         = False                             # False: reuse this state's mesh if it exists with the same sizes
FLUENT_VERSION = "26.1.0"                          # pinned (Decision 1)
PROCS          = 4                                 # Fluent processes

## 1. Imports

In [ ]:
import datetime
import json
import math
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2. Vishesh's pipeline modules
From `PIPELINE_ROOT`, unchanged: his lumen builder and stage helpers (`pipeline_cfd`), mesher, zone tools and launcher. Works with the `_vishesh` share copies and with his own repo. His launcher is told to use this notebook's Fluent version and process count.

In [ ]:
import sys
from types import SimpleNamespace

PIPE_ROOT = Path(PIPELINE_ROOT).resolve()
for sub in ("FullTwin", "StentForge", "StenosisCFD", "tools"):
    if (PIPE_ROOT / sub).is_dir() and str(PIPE_ROOT / sub) not in sys.path:
        sys.path.insert(0, str(PIPE_ROOT / sub))
os.environ["STENOSIS_FLUENT_PRODUCT_VERSION"] = FLUENT_VERSION     # read by his launcher at import
os.environ["STENOSIS_FLUENT_NPROC"] = str(PROCS)

def his(name):                                        # cfd_mesh_vishesh (share copy) or cfd_mesh (his repo)
    try:
        return __import__(f"{name}_vishesh")
    except ImportError:
        return __import__(name)

PC = his("pipeline_cfd")          # build_lumen, CSV_COLS, classify_case, merge_rows
VC = his("cfd_config")            # MeshControls
VM = his("cfd_mesh")              # watertight workflow tasks, mesh report, write
VR = his("cfd_run")               # zone split / centroid / validate
VL = his("fluent_launch")         # launcher with the licence-seat retry
MESH_YAML = next(p for p in (PIPE_ROOT / "StenosisCFD" / "config" / "cfd_pipeline_v3_vishesh.yaml",
                             PIPE_ROOT / "StenosisCFD" / "config" / "cfd_pipeline_v3.yaml") if p.exists())
PIPE_YAML = __import__("yaml").safe_load(MESH_YAML.read_text())
print(f"pipeline from {PIPE_ROOT}; launcher: Fluent {VL.PRODUCT_VERSION}, {VL.PROCESSOR_COUNT} process(es)")

## 3. Drop-in folders: inlet UDF and TAWSS / OSI
`dropin/inlet_udf/` with a `.c` file: one **transient** run at the waveform's mean velocity (`mean_velocity_mps`), the UDF's profile on the inlet, time settings from `inlet_udf.csv`; the 0.10 / 0.26 steady pair is not run. `dropin/tawss_osi/` with a `.c` file as well: compiled into the same library, 5 user memories, averaging reset at the start of the last cycle; TAWSS, OSI and RRT on the artery wall over the segment, written to `dropin/tawss_osi/tawss_osi_results.csv`. No `.c` in a folder: that part is off (both off = the steady parabolic run).

In [ ]:
DROP = Path(DROPIN_DIR).resolve()
INLET_C = sorted((DROP / "inlet_udf").glob("*.c"))
OSI_C = sorted((DROP / "tawss_osi").glob("*.c"))
TRANSIENT, OSI_ON = bool(INLET_C), bool(OSI_C)
TAWSS_OSI, T_RESET = None, None

def read_kv(path, need):
    d = pd.read_csv(path, comment="#", dtype=str, keep_default_na=False)
    kv = {str(k).strip(): str(v).strip() for k, v in zip(d["key"], d["value"]) if str(k).strip()}
    miss = [k for k in need if not kv.get(k)]
    if miss:
        raise ValueError(f"{path}: missing keys {miss}")
    return kv

def tui(cmd):                                         # a journal line or Scheme, as in the console (osi_setup.ipynb)
    return solver.execute_tui(cmd)

if OSI_ON and not TRANSIENT:
    raise RuntimeError("dropin/tawss_osi has a UDF but dropin/inlet_udf has none: TAWSS / OSI need the pulsatile (transient) inlet")
if TRANSIENT:
    UDF = read_kv(DROP / "inlet_udf" / "inlet_udf.csv",
                  ["udf_file", "profile_name", "period_s", "time_step_s", "cycles", "iterations_per_step", "mean_velocity_mps"])
    if not (DROP / "inlet_udf" / UDF["udf_file"]).exists():
        raise FileNotFoundError(f"inlet_udf.csv names {UDF['udf_file']}, which is not in {DROP / 'inlet_udf'}")
    PERIOD, DT = float(UDF["period_s"]), float(UDF["time_step_s"])
    STEPS = int(round(PERIOD / DT))
    if abs(STEPS * DT - PERIOD) > 1e-6 * PERIOD:
        raise ValueError(f"time_step_s {DT} does not divide period_s {PERIOD}")
    CYCLES, NITER = int(UDF["cycles"]), int(UDF["iterations_per_step"])
    PERIODIC_TOL = float(UDF.get("periodic_tol") or 0.01)
    INLET_U = float(UDF["mean_velocity_mps"])        # one run at the waveform mean replaces the steady pair
    U_LIST = f"{INLET_U:g}"
if OSI_ON:
    OSIS = read_kv(DROP / "tawss_osi" / "tawss_osi.csv", ["osi_threshold"])
    OSI_THR = float(OSIS["osi_threshold"])
    OSI_RESULTS = DROP / "tawss_osi" / "tawss_osi_results.csv"
print("inlet:", f"UDF {[f.name for f in INLET_C]}, transient: {CYCLES} cycles of {PERIOD} s, dt {DT} s "
      f"({STEPS} steps per cycle, {NITER} iterations per step), mean {INLET_U} m/s" if TRANSIENT else "steady parabolic (no inlet UDF)")
print("TAWSS / OSI:", f"UDF {[f.name for f in OSI_C]}, OSI threshold {OSI_THR}" if OSI_ON else "off (no UDF in dropin/tawss_osi)")

## 4. Lumen from the mechanical campaign
Vishesh's `build_lumen`, unchanged: the lineage's analytic L7 lumen for `blocked`; the FEA extract `<HELD_ROOT>/<DESIGN>/<STATE>/art_inner.npz` for `expanded` / `released` (stops with `FileNotFoundError` if it is missing). Extended lumen STEP + STL in the stage folder, built once per design and state; the other inflow reuses it. Picks the setup CSV for the domain.

In [ ]:
if STATE not in ("blocked", "expanded", "released"):
    raise ValueError(f"STATE must be blocked, expanded or released, got {STATE!r}")
U_ALL = sorted((float(u) for u in str(U_LIST).split(",")), reverse=True)
if not any(abs(INLET_U - u) < 1e-9 for u in U_ALL):
    raise ValueError(f"INLET_U = {INLET_U} is not in U_LIST = {U_LIST}")
STAGE_DIR = (Path(CAMPAIGN_OUT) / DESIGN / f"{STATE}_struts-no").resolve()
GEOM_JSON = STAGE_DIR / "geometry" / "geometry.json"

if LUMEN_STEP:                                    # test only
    LUMEN_REC = {"profile_source": "LUMEN_STEP (test, no build_lumen)", "step": str(Path(LUMEN_STEP).resolve())}
    EXT_MM = None
else:
    if GEOM_JSON.exists() and Path(json.loads(GEOM_JSON.read_text())["lumen"]["step_path"]).exists():
        LUMEN_REC = json.loads(GEOM_JSON.read_text())
        print(f"reusing the lumen built for {DESIGN} / {STATE}: {GEOM_JSON}")
    else:
        held = HELD_ROOT or str(PIPE_ROOT / "out" / "physics_v3" / "fallback" / "held")
        A = SimpleNamespace(design=DESIGN, state=STATE, struts="no", u=U_ALL, lineage=LINEAGE,
                            held_root=str(Path(held).resolve()))
        LUMEN_REC = PC.build_lumen(A, STAGE_DIR / "geometry")       # STEP + STL + geometry.json
    LUMEN_STEP = LUMEN_REC["lumen"]["step_path"]
    EXT_MM = float(LUMEN_REC["extension_per_side_mm"])

SETUP_CSV = "case_setup_L7_v3.csv" if STATE == "blocked" else "case_setup_L7_v3_quarter.csv"
OUT_DIR = str(STAGE_DIR)
print(f"{DESIGN} / {STATE} at {INLET_U} m/s: lumen {Path(LUMEN_STEP).name}; setup {SETUP_CSV}; out {STAGE_DIR}")

## 5. Read and check the setup CSV
Every key has a parser. An unknown, duplicate or missing key stops here, before Fluent starts. The segment starts at the extension `build_lumen` used (Vishesh's planes), which replaces `segment_z_start`.

In [ ]:
def f_(v):  return float(v)
def i_(v):  return int(float(v))
def s_(v):  return str(v).strip()
def yes_(v): return str(v).strip().lower() in ("yes", "y", "true", "1")
def list_(v): return [p.strip() for p in str(v).split(";") if p.strip()]
def flist_(v): return [float(p) for p in list_(v)]
def vec_(v):
    out = flist_(v)
    if len(out) != 3:
        raise ValueError(f"expected 3 components separated by ';', got {v!r}")
    return out
def one_of(*opts):
    def p(v):
        v = str(v).strip().lower()
        if v not in opts:
            raise ValueError(f"{v!r} is not one of {opts}")
        return v
    return p

# key: (parser, required)
SPEC = {
    "density": (f_, True), "viscosity_model": (one_of("carreau", "newtonian"), True),
    "carreau_mu0": (f_, True), "carreau_mu_inf": (f_, True),
    "carreau_lambda": (f_, True), "carreau_n": (f_, True),
    "mesh_units": (one_of("m", "mm"), True), "R0": (f_, True), "sector_fraction": (f_, True),
    "axis_origin": (vec_, True), "axis_direction": (vec_, True), "stenosis_area": (f_, True),
    "zone_inlet": (list_, True), "zone_outlet": (list_, True), "zone_wall": (list_, True),
    "zone_symmetry": (list_, False),
    "inlet_mean_velocity": (flist_, True), "inlet_profile": (one_of("parabolic", "uniform", "udf"), True),
    "outlet_gauge_pressure": (f_, True),
    "viscous_model": (one_of("laminar", "sst"), True),
    "inlet_turb_intensity": (f_, True), "inlet_hydraulic_diameter": (f_, True),
    "time": (one_of("steady", "transient"), True),
    "p_v_coupling": (s_, True), "gradient": (s_, True),
    "disc_pressure": (s_, True), "disc_convective": (s_, True),
    "residual_target": (f_, True), "max_iterations": (i_, True),
    "dp_flat_window": (i_, True), "dp_flat_tol": (f_, True),
    "stability_tol": (f_, True), "stability_window": (f_, True), "mass_imbalance_tol": (f_, True),
    "inlet_area_tol": (f_, True), "inlet_mass_flow_tol": (f_, True),
    "wss_sampling": (one_of("face", "node"), True), "low_wss_thresholds": (flist_, True),
    "segment_z_start": (f_, True), "segment_length": (f_, True),
    "bc_stage": (s_, False), "export_field": (yes_, False),
}

raw = pd.read_csv(SETUP_CSV, comment="#", dtype=str, keep_default_na=False)
raw["key"] = raw["key"].str.strip()
raw = raw[raw["key"] != ""]
problems = []
dups = raw["key"][raw["key"].duplicated()].tolist()
if dups:
    problems.append(f"duplicate keys: {dups}")
unknown = [k for k in raw["key"] if k not in SPEC]
if unknown:
    problems.append(f"unknown keys: {unknown}")
cfg = {}
for k, v in zip(raw["key"], raw["value"]):
    if k not in SPEC:
        continue
    parser, _ = SPEC[k]
    if str(v).strip() == "":
        cfg[k] = [] if parser in (list_, flist_) else None
        continue
    try:
        cfg[k] = parser(v)
    except Exception as e:
        problems.append(f"{k} = {v!r}: {e}")
missing = [k for k, (_, req) in SPEC.items() if req and cfg.get(k) in (None, [])]
if missing:
    problems.append(f"missing required keys: {missing}")
if problems:
    raise ValueError("case_setup problems:\n  " + "\n  ".join(problems))

cfg.setdefault("zone_symmetry", [])
cfg.setdefault("export_field", False)
if cfg["time"] != "steady":
    raise NotImplementedError("keep time = steady in the CSV: the transient run comes from dropin/inlet_udf")
if cfg["axis_origin"] != [0.0, 0.0, 0.0] or cfg["axis_direction"] != [0.0, 0.0, 1.0]:
    raise ValueError("this notebook assumes the pipeline frame: axis = +z through the origin")
if cfg["sector_fraction"] < 1 and not cfg["zone_symmetry"]:
    raise ValueError("sector_fraction < 1 but zone_symmetry is empty")
if cfg["wss_sampling"] != "face":
    raise NotImplementedError("wss_sampling = node is not wired; Decision 4 is face values, area-weighted")
if len(cfg["zone_inlet"]) != 1 or len(cfg["zone_outlet"]) != 1:
    raise ValueError("exactly one inlet zone and one outlet zone are expected")

ZONE_INLET, ZONE_OUTLET = cfg["zone_inlet"][0], cfg["zone_outlet"][0]
ZONE_WALL, ZONE_SYM = cfg["zone_wall"], cfg["zone_symmetry"]
print(f"viscous model: {cfg['viscous_model']}   inlet: {cfg['inlet_profile']}   "
      f"velocities in the CSV: {cfg['inlet_mean_velocity']} m/s")
if TRANSIENT and not any(abs(INLET_U - u) < 1e-9 for u in cfg["inlet_mean_velocity"]):
    cfg["inlet_mean_velocity"].append(INLET_U)      # the waveform mean from inlet_udf.csv
if EXT_MM is not None:
    cfg["segment_z_start"] = EXT_MM * 1e-3          # z = extension and extension + 25 mm
print(f"segment: z = {cfg['segment_z_start'] * 1e3:.2f} to {(cfg['segment_z_start'] + cfg['segment_length']) * 1e3:.2f} mm")
raw[["key", "value", "units", "source"]]

## 6. Flow numbers
Inlet flow and Reynolds numbers for this run's velocity.

In [ ]:
def carreau(gdot):
    return cfg["carreau_mu_inf"] + (cfg["carreau_mu0"] - cfg["carreau_mu_inf"]) * \
        (1 + (cfg["carreau_lambda"] * gdot) ** 2) ** ((cfg["carreau_n"] - 1) / 2)

def flow_numbers(U):
    if not any(abs(U - u) < 1e-9 for u in cfg["inlet_mean_velocity"]):
        raise ValueError(f"INLET_U = {U} is not in inlet_mean_velocity {cfg['inlet_mean_velocity']}")
    R0, rho = cfg["R0"], cfg["density"]
    A_nom = cfg["sector_fraction"] * math.pi * R0 ** 2
    gdot_w = 4 * U / R0                                   # Poiseuille wall shear rate
    mu_w = carreau(gdot_w) if cfg["viscosity_model"] == "carreau" else cfg["carreau_mu_inf"]
    return dict(
        U=U, A_inlet_nominal_m2=A_nom,
        Q_full_vessel_mL_min=U * math.pi * R0 ** 2 * 6e7,
        Re_mu_inf=rho * U * 2 * R0 / cfg["carreau_mu_inf"],
        Re_carreau_wall=rho * U * 2 * R0 / mu_w,
        throat_radius_mm=R0 * math.sqrt(1 - cfg["stenosis_area"]) * 1e3,
        U_throat_mean=U / (1 - cfg["stenosis_area"]),
    )

FLOW = flow_numbers(INLET_U)
for k, v in FLOW.items():
    print(f"  {k:22s} {v:.6g}")

## 7. Lumen geometry (no Fluent)
Reads the STL beside the STEP: length, inlet and throat radius, quarter or full, and the area of each face family (the expected areas for the zone check). Stops if the lumen disagrees with the CSV.

In [ ]:
def read_stl(path):
    b = Path(path).read_bytes()
    n = int(np.frombuffer(b, "<u4", 1, 80)[0])
    if 84 + 50 * n != len(b):
        raise ValueError(f"{path}: expected a binary STL")
    rec = np.frombuffer(b, np.dtype([("n", "<f4", 3), ("v", "<f4", (3, 3)), ("a", "<u2")]), n, 84)
    return rec["v"].astype(float)

STEP_PATH = Path(LUMEN_STEP).resolve()
STL_PATH = STEP_PATH.with_suffix(".stl")
for p in (STEP_PATH, STL_PATH):
    if not p.exists():
        raise FileNotFoundError(p)
T = read_stl(STL_PATH)                                    # mm
P = T.reshape(-1, 3)
L_mm = float(P[:, 2].max())
tol = 1e-6 * L_mm
tri_area = 0.5 * np.linalg.norm(np.cross(T[:, 1] - T[:, 0], T[:, 2] - T[:, 0]), axis=1)
fam = np.full(len(T), "wall", dtype=object)
fam[np.all(np.abs(T[:, :, 2]) < tol, 1)] = "inlet"
fam[np.all(np.abs(T[:, :, 2] - L_mm) < tol, 1)] = "outlet"
QUARTER = bool(P[:, 0].min() > -tol and P[:, 1].min() > -tol)
if QUARTER:
    fam[np.all(np.abs(T[:, :, 0]) < tol, 1)] = "symmetry"
    fam[np.all(np.abs(T[:, :, 1]) < tol, 1)] = "symmetry"
wall_pts = T[fam == "wall"].reshape(-1, 3)
r_wall = np.hypot(wall_pts[:, 0], wall_pts[:, 1])
inlet_pts = P[np.abs(P[:, 2]) < tol]
GEOM = dict(
    length_mm=L_mm, quarter=QUARTER,
    inlet_radius_mm=float(np.hypot(inlet_pts[:, 0], inlet_pts[:, 1]).max()),
    throat_radius_mm=float(r_wall.min()), throat_z_mm=float(wall_pts[r_wall.argmin(), 2]),
    expected_mm2={"inlet": float(tri_area[fam == "inlet"].sum()), "outlet": float(tri_area[fam == "outlet"].sum()),
                  "wall": float(tri_area[fam == "wall"].sum()),
                  **({"symmetry_each": float(tri_area[fam == "symmetry"].sum() / 2)} if QUARTER else {})},
)
problems = []
if QUARTER != (cfg["sector_fraction"] < 1):
    problems.append(f"lumen is {'quarter' if QUARTER else 'full 360'} but sector_fraction = {cfg['sector_fraction']}: use the matching CSV")
if abs(GEOM["inlet_radius_mm"] / (cfg["R0"] * 1e3) - 1) > 0.005:
    problems.append(f"inlet radius {GEOM['inlet_radius_mm']:.4f} mm vs R0 {cfg['R0'] * 1e3:.4f} mm")
seg0, seg1 = cfg["segment_z_start"] * 1e3, (cfg["segment_z_start"] + cfg["segment_length"]) * 1e3
if L_mm < seg1:
    problems.append(f"length {L_mm:.1f} mm does not fit the segment {seg0:.1f}-{seg1:.1f} mm")
if not (seg0 < GEOM["throat_z_mm"] < seg1):
    print(f"[warn] narrowest radius at z = {GEOM['throat_z_mm']:.2f} mm, outside the segment {seg0:.1f}-{seg1:.1f} mm")
if problems:
    raise ValueError("lumen and CSV disagree:\n  " + "\n  ".join(problems))
print(json.dumps(GEOM, indent=2))

## 8. Mesh sizes (Vishesh's throat-adapted rule)
Surface max size = throat radius ÷ `cells_per_throat_radius`, clamped to [floor, cap]; min = max ÷ `max_to_min_ratio`; growth, curvature angle and prism layers from the same block of his pipeline yaml (port of `cfd_v3_config_vishesh.mesh_controls_v3`).

In [ ]:
mb = PIPE_YAML["mesh"]
if mb["rule"] != "throat_adapted":
    raise ValueError(f"mesh.rule {mb['rule']!r}: only throat_adapted is implemented")
rt = GEOM["throat_radius_mm"]
max_mm = min(float(mb["max_size_cap_mm"]), max(float(mb["max_size_floor_mm"]), rt / float(mb["cells_per_throat_radius"])))
min_mm = max_mm / float(mb["max_to_min_ratio"])
MESH_CTRL = VC.MeshControls(
    min_size_mm=round(min_mm, 4), max_size_mm=round(max_mm, 4),
    growth_rate=float(mb["growth_rate"]), curvature_normal_angle=float(mb["curvature_normal_angle_deg"]),
    n_prism_layers=int(mb["prism_layers"]), prism_transition_ratio=float(mb["prism_transition_ratio"]),
    prism_growth_rate=float(mb["prism_growth_rate"]), volume_fill=str(mb["volume_fill"]))
print(f"throat radius {rt:.4f} mm -> surface max {MESH_CTRL.max_size_mm} mm, min {MESH_CTRL.min_size_mm} mm, "
      f"{MESH_CTRL.n_prism_layers} prism layers, {MESH_CTRL.volume_fill}")

## 9. Mesh in Fluent Meshing (Vishesh's watertight workflow)
His task functions, called in his order. The import asks for one zone per CAD face (port of `cfd_v2_vishesh._import_geometry_v2`: with one zone per body the extended lumen lost its inlet, measured 24 Sep). The mesh goes to the stage folder; the other inflow reuses it.

In [ ]:
STEP_PATH = Path(LUMEN_STEP).resolve()
MESH_DIR = Path(OUT_DIR) / "meshes"
MESH_DIR.mkdir(parents=True, exist_ok=True)
MESH_OUT = (MESH_DIR / f"{STEP_PATH.stem}_v3.msh.h5").resolve()
REPORT_PATH = MESH_DIR / f"{STEP_PATH.stem}_mesh_report.json"
SIZES = [MESH_CTRL.min_size_mm, MESH_CTRL.max_size_mm]
old = json.loads(REPORT_PATH.read_text()) if REPORT_PATH.exists() else None
if (not REMESH) and MESH_OUT.exists() and old is not None and old.get("sizes_mm") == SIZES:
    MESH_REPORT = old
    print(f"reusing {MESH_OUT.name} (same sizes {SIZES} mm; set REMESH = True to mesh again)")
else:
    t0 = time.time()
    meshing = VL.launch(mode="meshing", cwd=str(MESH_DIR.resolve()))
    try:
        meshing.workflow.InitializeWorkflow(WorkflowType="Watertight Geometry")
        task = VM._task(meshing, "Import Geometry")
        task.Arguments = dict(FileName=str(STEP_PATH), LengthUnit="mm", AppendMesh=False,
                              Directory=str(STEP_PATH.parent), CadImportOptions=dict(OneZonePer="face"))
        task.Execute()
        VM.add_local_sizing(meshing)
        VM.generate_surface_mesh(meshing, MESH_CTRL)
        VM.describe_geometry(meshing)
        VM.add_boundary_layers(meshing, MESH_CTRL)
        VM.generate_volume_mesh(meshing, MESH_CTRL)
        MESH_REPORT = VM.mesh_report(meshing)
        VM.write_mesh(meshing, str(MESH_OUT))
    finally:
        meshing.exit()
    MESH_REPORT.update(mesh_path=str(MESH_OUT), mesh_walltime_s=round(time.time() - t0, 1),
                       sizes_mm=SIZES, geometry=GEOM)
    print(json.dumps({k: v for k, v in MESH_REPORT.items() if k != "geometry"}, indent=2, default=str))

## 10. Mesh gates
Stops if the minimum orthogonal quality is below the target (0.1) or the cell count reaches the cap (512,000).

In [ ]:
cells, oq = MESH_REPORT.get("cell_count"), MESH_REPORT.get("min_orthogonal_quality")
if cells is None or oq is None:
    raise RuntimeError(f"mesh report incomplete: {MESH_REPORT}")
if oq < VC.MeshControls().min_orthogonal_quality_target:
    raise RuntimeError(f"min orthogonal quality {oq:.3f} < {VC.MeshControls().min_orthogonal_quality_target}")
if cells >= VC.MeshControls().student_cell_cap:
    raise RuntimeError(f"{cells} cells >= cap {VC.MeshControls().student_cell_cap}")
REPORT_PATH.write_text(json.dumps(MESH_REPORT, indent=2, default=str))
MESH_PATH = MESH_OUT
MESH_STEM = MESH_PATH.name.split(".")[0]
print(f"[mesh gates] {cells} cells, min orthogonal quality {oq:.3f}: OK -> {MESH_PATH}")

## 11. Launch Fluent 26.1
Vishesh's launcher, with his retry for the licence seat the mesher has just released.

In [ ]:
import ansys.fluent.core as pyfluent

out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)
solver = VL.launch_solver_with_retry(str(out.resolve()))
s = solver.settings
print(solver.get_fluent_version(), "| pyfluent", pyfluent.__version__)

## 12. Read the mesh

In [ ]:
s.file.read(file_type="mesh", file_name=str(MESH_PATH))
if cfg["mesh_units"] == "mm":
    s.mesh.scale(x_scale=1e-3, y_scale=1e-3, z_scale=1e-3)
s.setup.general.solver.type = "pressure-based"
s.setup.general.solver.time = "steady"
s.setup.models.energy.enabled = False
s.mesh.check()

## 13. Name the zones (Vishesh's method)
The meshed STEP arrives with generic wall zones. If the zone count differs from the CAD faces (3 full, 5 quarter), split at 40° (his `separate_boundary_zones`); classify each by its centroid (his rule); validate; check each zone's area against the STL within 3 % (his 27 Sep cap waiver for the caps); then retype and rename to `inlet`, `outlet`, `wall`, `sym-1`, `sym-2`.

In [ ]:
def zone_family(c, L_m, R0_m, quarter):
    xc, yc, zc = c
    if abs(zc) <= 0.02 * L_m:
        return "inlet"
    if abs(zc - L_m) <= 0.02 * L_m:
        return "outlet"
    if quarter and (abs(yc) <= 0.05 * R0_m or abs(xc) <= 0.05 * R0_m):
        return "symmetry"
    return "wall"

def zone_area_m2(z):
    return abs(float(solver.fields.reduction.area(locations=[z])))   # Vishesh's route (cfd_v2_vishesh._zone_area_m2); reports are inactive here

pipe = PIPE_YAML["zone_area"]
n_faces = 5 if GEOM["quarter"] else 3                # CAD faces: inlet, outlet, wall (+ 2 symmetry planes)
n_zones = len(VR._zones_by_type(solver))
if n_zones == n_faces:
    print(f"{n_zones} zones = {n_faces} CAD faces (one zone per face): no split needed")
else:
    print(f"{n_zones} zones for {n_faces} CAD faces: splitting at 40 deg (Vishesh's separate_boundary_zones)")
    VR.separate_boundary_zones(solver)
L_m, R0_m, Q = GEOM["length_mm"] * 1e-3, cfg["R0"], GEOM["quarter"]
ZMAP = {}
for z in VR._zones_by_type(solver):
    c = VR._zone_centroid(solver, z)
    if c is None:
        raise RuntimeError(f"no centroid for zone {z}")
    ZMAP[z] = zone_family(c, L_m, R0_m, Q)
ok, why = VR.validate_zone_map(ZMAP, Q)
if not ok:
    raise RuntimeError(f"zone classification failed: {why}; {ZMAP}")

exp = GEOM["expected_mm2"]
table, problems = {}, []
for z, f in ZMAP.items():
    a = zone_area_m2(z) * 1e6
    e = exp["symmetry_each" if f == "symmetry" else f]
    table[z] = dict(family=f, area_mm2=a, expected_mm2=e, rel_err=(a - e) / e)
    if abs(table[z]["rel_err"]) > 0.03:
        problems.append(f"{z} classified {f}: area {a:.4f} mm2 vs expected {e:.4f} ({100 * table[z]['rel_err']:+.1f} %)")
caps = {r["family"]: r["area_mm2"] for r in table.values() if r["family"] in ("inlet", "outlet")}
waived = []
for pr in list(problems):                 # 27 Sep cap waiver: caps agree within 0.5 % and sit within 10 % of expected
    if (" classified inlet:" in pr or " classified outlet:" in pr) and len(caps) == 2 \
            and abs(caps["inlet"] - caps["outlet"]) <= pipe["cap_pair_agreement"] * caps["inlet"] \
            and abs(caps["inlet"] - exp["inlet"]) <= pipe["cap_vs_expected"] * exp["inlet"]:
        problems.remove(pr)
        waived.append(pr)
print(pd.DataFrame(table).T.to_string())
if waived:
    print("[cap waiver]", waived)
if problems:
    raise RuntimeError("zone-area check failed (a zone is mis-classified):\n  " + "\n  ".join(problems))

sym_n = 0
for z, f in ZMAP.items():                 # retype, then rename (cfd_run_vishesh.setup_boundaries, rename part)
    ftype = {"inlet": "velocity-inlet", "outlet": "pressure-outlet", "symmetry": "symmetry", "wall": "wall"}[f]
    solver.tui.mesh.modify_zones.zone_type(z, ftype)
    if f == "symmetry":
        sym_n += 1
        new = f"sym-{sym_n}"
    else:
        new = f
    if new != z:
        solver.tui.mesh.modify_zones.zone_name(z, new)
ZONE_TABLE = table
print("zones renamed:", {z: f for z, f in ZMAP.items()})

## 14. Check the zones
Stops if a zone named in the CSV is missing or has the wrong type. Exact names; no wildcards.

In [ ]:
bc = s.setup.boundary_conditions
zones = {}
for ztype in ("velocity_inlet", "pressure_outlet", "wall", "symmetry", "interior"):
    for zname in getattr(bc, ztype).get_object_names():
        zones[zname] = ztype

expected = {ZONE_INLET: "velocity_inlet", ZONE_OUTLET: "pressure_outlet"}
expected.update({z: "wall" for z in ZONE_WALL})
expected.update({z: "symmetry" for z in ZONE_SYM})
wrong = [f"{z}: expected {t}, found {zones.get(z, 'missing')}" for z, t in expected.items() if zones.get(z) != t]
if wrong:
    raise ValueError("zone check failed:\n  " + "\n  ".join(wrong) + f"\nzones in the case: {zones}")
unused = [z for z, t in zones.items() if t != "interior" and z not in expected]
if unused:
    print(f"[warn] boundary zones not named in the CSV (left as they are): {unused}")

FLUID_ZONES = list(s.setup.cell_zone_conditions.fluid.get_object_names())
if not FLUID_ZONES:
    raise RuntimeError("no fluid cell zone in the case")
print("fluid cell zones:", FLUID_ZONES)
pd.DataFrame(sorted(zones.items()), columns=["zone", "type"])

## 15. Viscous model
Laminar (Decision 7 open). Set before the material: Fluent offers Carreau only once the flow model is laminar.

In [ ]:
viscous = s.setup.models.viscous
if cfg["viscous_model"] == "laminar":
    viscous.model = "laminar"
else:
    viscous.model = "k-omega"
    viscous.k_omega_model = "sst"
print("viscous model:", viscous.model())

## 16. Material: blood, Carreau
Stops if any constant Fluent holds differs from the CSV.

In [ ]:
fluids = s.setup.materials.fluid
if "blood" not in fluids.get_object_names():
    fluids.create("blood")
blood = fluids["blood"]
blood.density.option = "constant"
blood.density.value = cfg["density"]
if cfg["viscosity_model"] == "carreau":
    blood.viscosity.option = "carreau"
    blood.viscosity.carreau.zero_shear_viscosity = cfg["carreau_mu0"]
    blood.viscosity.carreau.infinite_shear_viscosity = cfg["carreau_mu_inf"]
    blood.viscosity.carreau.time_constant = cfg["carreau_lambda"]
    blood.viscosity.carreau.power_law_index = cfg["carreau_n"]
else:
    blood.viscosity.option = "constant"
    blood.viscosity.value = cfg["carreau_mu_inf"]

for z in FLUID_ZONES:
    s.setup.cell_zone_conditions.fluid[z].material = "blood"     # path checked in Task 0.3

rv = s.setup.reference_values
rv.density = cfg["density"]
rv.viscosity = cfg["carreau_mu_inf"]
rv.length = 2 * cfg["R0"]

print("density:", blood.density.value())
if cfg["viscosity_model"] == "carreau":
    rb = blood.viscosity.carreau.get_state()
    print(json.dumps(rb, indent=2, default=str))       # also shows any temperature term (should be 0 / off)
    want = {"zero_shear_viscosity": cfg["carreau_mu0"], "infinite_shear_viscosity": cfg["carreau_mu_inf"],
            "time_constant": cfg["carreau_lambda"], "power_law_index": cfg["carreau_n"]}
    bad = {k: (rb.get(k), v) for k, v in want.items() if rb.get(k) is None or abs(rb[k] - v) > 1e-6 * abs(v)}
    if bad or abs(blood.density.value() - cfg["density"]) > 1e-6 * cfg["density"]:
        raise RuntimeError(f"blood constants not as set (Fluent, CSV): {bad}")

## 17. UDFs from the drop-in folders
Only with a `.c` in `dropin/inlet_udf/`. The sources are copied to the stage folder (Fluent's working folder) and compiled into one library, `libudf`, with Fluent's built-in compiler. With the TAWSS / OSI UDF: 5 user memories, its initialisation and end-of-step hooks, and the wall zone it reads (`face-zone-walls` = the artery wall). The solver is switched to transient (2nd order implicit).

In [ ]:
if TRANSIENT:
    srcs = INLET_C + OSI_C
    for f in srcs:
        shutil.copy(f, out / f.name)
    if (out / "libudf").exists():
        shutil.rmtree(out / "libudf")
    ud = solver.tui.define.user_defined               # same calls as the osi_setup.ipynb example that comes with osi.c
    if OSI_ON:
        w0 = ZONE_WALL[0]                             # the wall the OSI UDF samples (its face-zone-walls variable)
        tui(f"(make-new-rpvar 'get-thread-by-name/face-zone-walls \"{w0}\" 'string)")
        tui(f"(rpsetvar 'get-thread-by-name/face-zone-walls \"{w0}\")")
        ud.user_defined_memory("5")
    ud.use_built_in_compiler("y")
    ud.compiled_functions("compile", "libudf", "yes", *[f.name for f in srcs], "", "")
    ud.compiled_functions("load", "libudf")
    if not (out / "libudf").exists():
        raise RuntimeError(f"UDF compile failed: no libudf folder in {out}; see the Fluent transcript there")
    if OSI_ON:
        tui('/define/user-defined/function-hooks/initialization "Initialise_UDMIs::libudf" ""')
        tui('/define/user-defined/function-hooks/execute-at-end "data_capture::libudf" ""')
    s.setup.general.solver.time = "unsteady-2nd-order"
    print(f"libudf compiled and loaded from {[f.name for f in srcs]}; solver time:", s.setup.general.solver.time())

## 18. Run tag and reports
Each run (one inflow) gets its own folder and monitor file in the stage folder.

In [ ]:
FLOW = flow_numbers(INLET_U)
TAG = f"{MESH_STEM}_{cfg['viscous_model']}_U{INLET_U:.2f}".replace(".", "p")
CASE_DIR = out / TAG
CASE_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(SETUP_CSV, CASE_DIR / "case_setup_used.csv")
print(f"==== {TAG} ====")

rd = s.solution.report_definitions
rd.surface["a_inlet"] = {}
rd.surface["a_inlet"].report_type = "surface-area"
rd.surface["a_inlet"].surface_names = [ZONE_INLET]

rd.flux["mflow_inlet"] = {}
rd.flux["mflow_inlet"].report_type = "flux-massflow"
rd.flux["mflow_inlet"].boundaries = [ZONE_INLET]
rd.flux["mflow_outlet"] = {}
rd.flux["mflow_outlet"].report_type = "flux-massflow"
rd.flux["mflow_outlet"].boundaries = [ZONE_OUTLET]

rd.surface["p_inlet"] = {}                       # = pressure drop (outlet is 0 Pa); used for dp-flat in Phase 1
rd.surface["p_inlet"].report_type = "surface-areaavg"
rd.surface["p_inlet"].field = "pressure"
rd.surface["p_inlet"].surface_names = [ZONE_INLET]

rd.surface["wss_wall_areaavg"] = {}
rd.surface["wss_wall_areaavg"].report_type = "surface-areaavg"
rd.surface["wss_wall_areaavg"].field = "wall-shear"
rd.surface["wss_wall_areaavg"].surface_names = ZONE_WALL
rd.surface["wss_wall_max"] = {}
rd.surface["wss_wall_max"].report_type = "surface-facetmax"
rd.surface["wss_wall_max"].field = "wall-shear"
rd.surface["wss_wall_max"].surface_names = ZONE_WALL

rd.volume["vmax"] = {}
rd.volume["vmax"].report_type = "volume-max"
rd.volume["vmax"].field = "velocity-magnitude"
rd.volume["vmax"].cell_zones = FLUID_ZONES

MON_DEFS = ["mflow_inlet", "mflow_outlet", "p_inlet", "wss_wall_areaavg", "wss_wall_max", "vmax"]
MON_FILE = (CASE_DIR / f"{TAG}_monitors.out").resolve()
rf = s.solution.monitor.report_files
rf["monitors"] = {}
rf["monitors"].report_defs = MON_DEFS
rf["monitors"].file_name = str(MON_FILE)
rf["monitors"].frequency = 1

def compute(names):
    vals = {}
    for item in s.solution.report_definitions.compute(report_defs=list(names)) or []:
        for k, v in item.items():
            vals[k] = float(v[0] if isinstance(v, (list, tuple)) else v)
    return vals

def read_monitor_file(path):
    rows, header = [], None
    if not Path(path).exists():
        return None, None
    for line in open(path):
        line = line.strip()
        if not line:
            continue
        if line.startswith("("):
            header = [h.strip('"') for h in line.strip("()").split('" "')]
            continue
        try:
            rows.append([float(v) for v in line.split()])
        except ValueError:
            continue
    return header, (np.array(rows) if rows else None)

## 19. Gate 1: inlet area
Catches a mm/m mix-up and a full/quarter mismatch before anything is solved.

In [ ]:
s.solution.initialization.standard_initialize()
A_IN = abs(compute(["a_inlet"])["a_inlet"])
ratio = A_IN / FLOW["A_inlet_nominal_m2"]
print(f"[gate 1] inlet area {A_IN:.6e} m2 vs nominal {FLOW['A_inlet_nominal_m2']:.6e} m2 (ratio {ratio:.4f})")
if abs(ratio - 1) > cfg["inlet_area_tol"]:
    hint = "check R0, sector_fraction, mesh_units and zone_inlet"
    if 0.5e6 < ratio < 2e6:
        hint = "the mesh looks like it is in mm: set mesh_units = mm"
    elif 0.5e-6 < ratio < 2e-6:
        hint = "the mesh is scaled but already in m: set mesh_units = m"
    elif abs(ratio * cfg["sector_fraction"] - 1) < 0.05:
        hint = "the mesh looks like a full vessel: set sector_fraction = 1"
    raise RuntimeError(f"inlet area gate failed: {hint}")
R_IN = math.sqrt(A_IN / (cfg["sector_fraction"] * math.pi))

## 20. Boundary conditions
The only cell that sets the inlet: parabolic with the nominal R0, as the pipeline. Harshita's BC module (H3) hooks in here later.

In [ ]:
if cfg.get("bc_stage"):
    raise NotImplementedError("bc_stage (Harshita's H3 module) is not wired yet; leave it blank")

inlet = bc.velocity_inlet[ZONE_INLET]
inlet.momentum.velocity_specification_method = "Magnitude, Normal to Boundary"
if TRANSIENT:
    UDF_NAME = f"{UDF['profile_name']}::libudf"
    INLET_EXPR, tried = f"UDF {UDF_NAME}", []
    for how in ("settings", "settings-option", "settings-attrs", "tui"):
        try:
            if how == "settings-option":                 # the form in the osi_setup.ipynb example that comes with osi.c
                inlet.momentum.velocity_magnitude = {"option": f"udf {UDF_NAME}"}
            elif how == "settings":
                inlet.momentum.velocity_magnitude = {"option": "udf", "udf": UDF_NAME}
            elif how == "settings-attrs":
                inlet.momentum.velocity_magnitude.option = "udf"
                inlet.momentum.velocity_magnitude.udf = UDF_NAME
            else:
                tui(f'/define/boundary-conditions/set/velocity-inlet {ZONE_INLET} () vmag yes "udf" "{UDF_NAME}" q')
            if UDF["profile_name"] in str(inlet.momentum.velocity_magnitude.get_state()):
                print(f"inlet UDF hooked ({how}):", inlet.momentum.velocity_magnitude.get_state())
                break
            tried.append(f"{how}: read back {inlet.momentum.velocity_magnitude.get_state()}")
        except Exception as e:
            tried.append(f"{how}: {type(e).__name__}: {e}")
    else:
        raise RuntimeError("could not put the inlet UDF on the inlet: " + " | ".join(tried))
elif cfg["inlet_profile"] == "parabolic":
    # radius = nominal R0 from the CSV: the same expression as the pipeline / 08_cfd cases (Task 0.3, 2 Oct)
    INLET_EXPR = (f"max(0[m/s], 2*{INLET_U:.10g}[m/s]*"
                  f"(1-(Position.x^2+Position.y^2)/({cfg['R0']:.10g}[m])^2))")
    s.setup.named_expressions["u_inlet"] = {"definition": INLET_EXPR}
    inlet.momentum.velocity_magnitude = "u_inlet"
elif cfg["inlet_profile"] == "uniform":
    INLET_EXPR = None
    inlet.momentum.velocity_magnitude = INLET_U
else:
    raise NotImplementedError("inlet_profile = udf: the v3 pulsatile UDF is hooked here with H3")
if cfg["viscous_model"] == "sst":
    inlet.turbulence.turbulence_specification = "Intensity and Hydraulic Diameter"   # name checked in Task 0.3
    inlet.turbulence.turbulent_intensity = cfg["inlet_turb_intensity"]
    inlet.turbulence.hydraulic_diameter = cfg["inlet_hydraulic_diameter"]

outlet = bc.pressure_outlet[ZONE_OUTLET]
outlet.momentum.gauge_pressure = cfg["outlet_gauge_pressure"]
if cfg["viscous_model"] == "sst":
    outlet.turbulence.turbulence_specification = "Intensity and Hydraulic Diameter"
    outlet.turbulence.backflow_turbulent_intensity = cfg["inlet_turb_intensity"]
    outlet.turbulence.backflow_hydraulic_diameter = cfg["inlet_hydraulic_diameter"]

s.setup.reference_values.velocity = INLET_U
print("inlet:", INLET_EXPR or f"uniform {INLET_U} m/s")

## 21. Numerics
Residual criterion type `absolute` on Fluent's (default) scaled residuals, as in Vishesh's pipeline.

In [ ]:
methods = s.solution.methods
methods.p_v_coupling.flow_scheme = cfg["p_v_coupling"]
methods.spatial_discretization.gradient_scheme = cfg["gradient"]
methods.spatial_discretization.discretization_scheme["pressure"] = cfg["disc_pressure"]
methods.spatial_discretization.discretization_scheme["mom"] = cfg["disc_convective"]
if cfg["viscous_model"] == "sst":
    methods.spatial_discretization.discretization_scheme["k"] = cfg["disc_convective"]
    methods.spatial_discretization.discretization_scheme["omega"] = cfg["disc_convective"]

residual = s.solution.monitor.residual
residual.options.criterion_type = "absolute"
for eq in residual.equations.get_object_names():
    residual.equations[eq].check_convergence = True
    residual.equations[eq].absolute_criteria = cfg["residual_target"]

print("coupling:", methods.p_v_coupling.flow_scheme())
print("schemes:", methods.spatial_discretization.discretization_scheme.get_state())
print("residual equations:", list(residual.equations.get_object_names()))

## 22. Initialise, Gate 2: inlet mass flow
Target = ρ·U·sector·π·R0²; tolerance from the CSV.

In [ ]:
s.solution.initialization.hybrid_initialize()
MDOT_TARGET = cfg["density"] * INLET_U * FLOW["A_inlet_nominal_m2"]     # v3 nominal flow (sector * pi * R0^2)
if OSI_ON:
    tui('/define/user-defined/execute-on-demand "get_thread_ids::libudf"')   # the wall zone the OSI UDF samples
if TRANSIENT:
    print("[gate 2] transient run: checked on the cycle-mean flow after the solve")
else:
    MDOT_INIT = abs(compute(["mflow_inlet"])["mflow_inlet"])
    err = abs(MDOT_INIT - MDOT_TARGET) / MDOT_TARGET
    print(f"[gate 2] inlet mass flow {MDOT_INIT:.6e} kg/s vs rho*U*pi*R0^2 {MDOT_TARGET:.6e} kg/s ({err:.3%})")
    if err > cfg["inlet_mass_flow_tol"]:
        raise RuntimeError("inlet mass-flow gate failed: check the axis, R0, mesh_units and sector_fraction")

## 23. Solve
Iterates to the residual target, then in batches of 10 until the inlet pressure (= Δp) is flat within `dp_flat_tol` over the last `dp_flat_window` iterations, or the iteration cap.

In [ ]:
def dp_history():
    hdr, mon = read_monitor_file(MON_FILE)
    return np.array([]) if mon is None else mon[:, hdr.index("p_inlet")]

if TRANSIENT:
    try:
        rf["monitors"].frequency_of = "time-step"
    except Exception as e:
        print("[note] report file frequency:", e)
    Z0, Z1 = cfg["segment_z_start"], cfg["segment_z_start"] + cfg["segment_length"]
    planes = s.results.surfaces.plane_surface
    for name, z in (("seg_up", Z0), ("seg_dn", Z1)):         # segment planes now, so they are recorded every step
        planes[name] = {}
        planes[name].method = "xy-plane"
        planes[name].z = z
        rd.surface[f"p_{name}"] = {}
        rd.surface[f"p_{name}"].report_type = "surface-areaavg"
        rd.surface[f"p_{name}"].field = "pressure"
        rd.surface[f"p_{name}"].surface_names = [name]
    rf["monitors"].report_defs = MON_DEFS + ["p_seg_up", "p_seg_dn"]
    tc = s.solution.run_calculation.transient_controls    # as the osi_setup.ipynb example that comes with osi.c
    tc.time_step_size = DT
    tc.time_step_count = STEPS                             # one cycle per dual_time_iterate()
    tc.max_iter_per_time_step = NITER
    t0 = time.time()
    for cyc in range(CYCLES):
        if OSI_ON and cyc == CYCLES - 1:                   # TAWSS / OSI over the last cycle: zero the averages and
            tui('/define/user-defined/execute-on-demand "reset_averaging::libudf"')   # the flow time (the UDF
            tui("(rpsetvar 'flow-time 0.0)")                # averages over flow time; a whole number of cycles
            tui("(rpsetvar 'time-step 0)")                  # keeps the inlet phase)
            T_RESET = cyc * PERIOD
        s.solution.run_calculation.dual_time_iterate()
        print(f"cycle {cyc + 1}/{CYCLES} done ({round(time.time() - t0)} s)")
    WALL_TIME = round(time.time() - t0, 1)

    hdr, mon = read_monitor_file(MON_FILE)
    if "flow-time" in hdr:                                    # one row per time step (last row of each step)
        ft = mon[:, hdr.index("flow-time")]
        keep = np.r_[ft[1:] != ft[:-1], True]
        mon, ft = mon[keep], ft[keep]
    mcol = lambda n: mon[:, hdr.index(n)]           # last-cycle columns (cell 24 reuses the name col)
    last, prev = slice(-STEPS, None), slice(-2 * STEPS, -STEPS)
    DP_CYCLE = float(np.mean(mcol("p_inlet")[last]))
    DP_SEG_CYCLE = float(np.mean(mcol("p_seg_up")[last] - mcol("p_seg_dn")[last]))
    DP_PREV = float(np.mean(mcol("p_inlet")[prev])) if len(mon) >= 2 * STEPS else None
    PERIODIC_CHANGE = abs(DP_CYCLE - DP_PREV) / abs(DP_CYCLE) if DP_PREV is not None else None
    PERIODIC_OK = PERIODIC_CHANGE is not None and PERIODIC_CHANGE <= PERIODIC_TOL
    MDOT_TARGET = cfg["density"] * INLET_U * FLOW["A_inlet_nominal_m2"]
    MDOT_CYCLE = float(np.mean(np.abs(mcol("mflow_inlet")[last])))
    MDOT_INIT = MDOT_CYCLE
    err = abs(MDOT_CYCLE - MDOT_TARGET) / MDOT_TARGET
    print(f"[gate 2] cycle-mean inlet mass flow {MDOT_CYCLE:.6e} kg/s vs rho*U*A {MDOT_TARGET:.6e} kg/s ({err:.3%})")
    if err > cfg["inlet_mass_flow_tol"]:
        raise RuntimeError("inlet mass-flow gate failed on the cycle mean: check mean_velocity_mps against the UDF")
    p, BATCHES, DP_SPAN, DP_FLAT = mcol("p_inlet"), 0, PERIODIC_CHANGE, PERIODIC_OK
    print(f"last cycle: mean dp {DP_CYCLE:.3f} Pa, segment {DP_SEG_CYCLE:.3f} Pa; change from the cycle before "
          f"{PERIODIC_CHANGE if PERIODIC_CHANGE is None else f'{PERIODIC_CHANGE:.3%}'} (periodic within {PERIODIC_TOL:.1%}: {PERIODIC_OK}); "
          f"wall time {WALL_TIME} s")
else:
    MAX_IT, W, TOL = cfg["max_iterations"], cfg["dp_flat_window"], cfg["dp_flat_tol"]
    t0 = time.time()
    s.solution.run_calculation.iterate(iter_count=MAX_IT)
    BATCHES = 0
    while True:
        p = dp_history()
        DP_SPAN = float(p[-(W + 1):].max() - p[-(W + 1):].min()) if len(p) > W else None
        DP_FLAT = DP_SPAN is not None and DP_SPAN <= TOL
        if DP_FLAT or len(p) >= MAX_IT:
            break
        s.solution.run_calculation.iterate(iter_count=min(10, MAX_IT - len(p)))
        BATCHES += 1
        if len(dp_history()) == len(p):
            print("[warn] Fluent ran no further iterations; leaving the dp-flat loop")
            break
    WALL_TIME = round(time.time() - t0, 1)
    print(f"iterations {len(p)}; extra batches {BATCHES}; dp span over the last {W}: "
          f"{DP_SPAN if DP_SPAN is None else round(DP_SPAN, 5)} Pa (flat: {DP_FLAT}); wall time {WALL_TIME} s")

## 24. Status
CONVERGED only if every residual is at or below `residual_target` AND Δp is flat. Monitor spread and mass imbalance are reported.

In [ ]:
VALS = compute(MON_DEFS)
it, res_data = solver.monitors.get_monitor_set_data(monitor_set_name="residual")
RES_LAST = {k: float(np.asarray(v)[-1]) for k, v in res_data.items()}
HDR, MON = read_monitor_file(MON_FILE)
N_IT = int(len(MON)) if MON is not None else None

SPREAD = {}
if MON is not None and len(MON) > 20:
    w = max(10, int(cfg["stability_window"] * len(MON)))
    for j, nm in enumerate(HDR[1:MON.shape[1]]):
        col = MON[-w:, 1 + j]
        SPREAD[nm] = float((col.max() - col.min()) / max(abs(col[-1]), 1e-30))

mi, mo = VALS["mflow_inlet"], VALS["mflow_outlet"]
IMBALANCE = abs(mi + mo) / abs(mi)
RES_OK = all(v <= cfg["residual_target"] for v in RES_LAST.values())
SPREAD_OK = bool(SPREAD) and all(v <= cfg["stability_tol"] for v in SPREAD.values())
MASS_OK = IMBALANCE <= cfg["mass_imbalance_tol"]
CONVERGED = RES_OK and DP_FLAT
if CONVERGED:
    STATUS = "CONVERGED"
elif RES_OK:
    STATUS = "RESIDUALS MET, DP NOT FLAT"
elif DP_FLAT:
    STATUS = "DP FLAT, RESIDUALS ABOVE TARGET"
else:
    STATUS = "NOT CONVERGED"
if TRANSIENT:                                        # residuals are per time step; the verdict is cycle-to-cycle
    CONVERGED = PERIODIC_OK
    STATUS = "CONVERGED (periodic)" if PERIODIC_OK else "NOT PERIODIC"
print(f"[{TAG}] {STATUS}; iterations {N_IT}; residuals {RES_LAST}")
print(f"  reported: spread ok {SPREAD_OK}; mass imbalance {IMBALANCE:.2e} (ok {MASS_OK})")
print(f"  dp (full) {VALS['p_inlet']:.3f} Pa; vmax {VALS['vmax']:.4f} m/s")

## 25. Segment Δp
Area-averaged pressure on planes at the two ends of the 25 mm artery segment.

In [ ]:
if TRANSIENT:                                        # planes made before the solve; mean over the last cycle
    Z0, Z1 = cfg["segment_z_start"], cfg["segment_z_start"] + cfg["segment_length"]
    SEG_P = {"p_seg_up": float(np.mean(mcol("p_seg_up")[last])), "p_seg_dn": float(np.mean(mcol("p_seg_dn")[last]))}
    DP_SEG = DP_SEG_CYCLE
    print(f"segment z = {Z0 * 1e3:.1f} to {Z1 * 1e3:.1f} mm: last-cycle mean dp_segment {DP_SEG:.3f} Pa")
else:
    Z0 = cfg["segment_z_start"]
    Z1 = Z0 + cfg["segment_length"]
    planes = s.results.surfaces.plane_surface
    for name, z in (("seg_up", Z0), ("seg_dn", Z1)):
        planes[name] = {}
        planes[name].method = "xy-plane"
        planes[name].z = z
        rd.surface[f"p_{name}"] = {}
        rd.surface[f"p_{name}"].report_type = "surface-areaavg"
        rd.surface[f"p_{name}"].field = "pressure"
        rd.surface[f"p_{name}"].surface_names = [name]
    SEG_P = compute(["p_seg_up", "p_seg_dn"])
    DP_SEG = SEG_P["p_seg_up"] - SEG_P["p_seg_dn"]
    print(f"segment z = {Z0 * 1e3:.1f} to {Z1 * 1e3:.1f} mm: p_up {SEG_P['p_seg_up']:.3f} Pa, "
          f"p_dn {SEG_P['p_seg_dn']:.3f} Pa, dp_segment {DP_SEG:.3f} Pa")

## 26. Wall shear stress on the artery wall
Face values with face areas, exported at face centres: faces whose centre lies in the segment, area-weighted mean, maximum, and the area fraction below each `low_wss_thresholds` value. Other wall zones are reported separately.

In [ ]:
WALL_ARTERY = ZONE_WALL[0]

def wall_faces(zone):
    fn = (CASE_DIR / f"{TAG}_faces_{zone}.csv").resolve()
    s.file.export.ascii(file_name=str(fn), surface_name_list=[zone], delimiter="comma",
                        quantities=["wall-shear", "x-wall-shear", "y-wall-shear", "z-wall-shear", "face-area-magnitude"],
                        location="cell-center")
    d = pd.read_csv(fn)
    d.columns = [c.strip() for c in d.columns]
    need = {"z-coordinate", "wall-shear", "face-area-magnitude"}
    if not need <= set(d.columns):
        raise RuntimeError(f"face export is missing {need - set(d.columns)}; it has {list(d.columns)}")
    return d

def wss_stats(d, zlo, zhi):
    m = (d["z-coordinate"] >= zlo) & (d["z-coordinate"] <= zhi)
    w = d.loc[m, "wall-shear"].to_numpy()
    a = d.loc[m, "face-area-magnitude"].to_numpy()
    out = dict(n_faces=int(m.sum()), area_m2=float(a.sum()), wss_mean_pa=float(np.average(w, weights=a)),
               wss_max_pa=float(w.max()), wss_min_pa=float(w.min()))
    for t in cfg["low_wss_thresholds"]:
        out[f"low_wss_frac_lt_{t:g}"] = float(a[w < t].sum() / a.sum())
    return out

FACES = {z: wall_faces(z) for z in ZONE_WALL}
WSS_SEG = wss_stats(FACES[WALL_ARTERY], Z0, Z1)
WSS_WALL = wss_stats(FACES[WALL_ARTERY], -np.inf, np.inf)
WSS_STRUT = {z: wss_stats(d, -np.inf, np.inf) for z, d in FACES.items() if z != WALL_ARTERY}

# cross-check: Fluent's own area-weighted average over the same wall should match the whole-wall mean
if len(ZONE_WALL) == 1:
    diff = abs(WSS_WALL["wss_mean_pa"] - VALS["wss_wall_areaavg"]) / VALS["wss_wall_areaavg"]
    print(f"[check] whole-wall mean {WSS_WALL['wss_mean_pa']:.5f} Pa vs Fluent area-average "
          f"{VALS['wss_wall_areaavg']:.5f} Pa ({diff:.3%})")
    if diff > 0.005:
        print("[warn] face export and Fluent's report differ by more than 0.5 %: check the export")
if TRANSIENT:
    print("[note] transient run: the face WSS above is the instant at the end of the last cycle")
pd.DataFrame({"segment (artery)": WSS_SEG, "whole artery wall": WSS_WALL, **WSS_STRUT})

## 27. TAWSS, OSI and RRT (drop-in)
Only with the TAWSS / OSI UDF. Its memories (TAWSS, OSI) are exported on the artery wall faces with the face areas. They hold the last cycle only: the averages and the flow time were reset at its start (as in the `osi_setup.ipynb` example that comes with `osi.c`). RRT = 1 / ((1 − 2 OSI) TAWSS). Over the 25 mm artery segment, area-weighted, as Vishesh's `wss_report_v3`: TAWSS mean and max, area fraction with TAWSS below each `low_wss_thresholds` value, OSI mean and max, area fraction with OSI above `osi_threshold`, RRT mean (finite faces) and 90th percentile. One row per design and state in `dropin/tawss_osi/tawss_osi_results.csv`.

In [ ]:
if OSI_ON:
    fn = (CASE_DIR / f"{TAG}_tawss_osi_{WALL_ARTERY}.csv").resolve()
    tried = []
    for names in (["tawss", "osi"], ["udm-3", "udm-4"], ["user-memory-3", "user-memory-4"]):
        try:
            s.file.export.ascii(file_name=str(fn), surface_name_list=[WALL_ARTERY], delimiter="comma",
                                quantities=names + ["face-area-magnitude"], location="cell-center")
            d = pd.read_csv(fn)
            d.columns = [c.strip() for c in d.columns]
            if all(n in d.columns for n in names):
                break
            tried.append(f"{names}: columns {list(d.columns)}")
        except Exception as e:
            tried.append(f"{names}: {type(e).__name__}: {e}")
    else:
        raise RuntimeError("could not export the TAWSS / OSI memories: " + " | ".join(tried))
    T_END = CYCLES * PERIOD
    scale = 1.0                                        # flow time was reset with the averages: no rescaling
    m = (d["z-coordinate"] >= Z0) & (d["z-coordinate"] <= Z1)
    tw = d.loc[m, names[0]].to_numpy() * scale
    os_ = d.loc[m, names[1]].to_numpy()
    a = d.loc[m, "face-area-magnitude"].to_numpy()
    den = (1 - 2 * os_) * tw
    rrt = np.where(den > 0, 1 / np.where(den > 0, den, 1), np.inf)
    fin = np.isfinite(rrt)
    order = np.argsort(rrt, kind="stable")
    cum = np.cumsum(a[order])
    TAWSS_OSI = {"design": DESIGN, "state": STATE, "U_mean_mps": INLET_U, "period_s": PERIOD, "time_step_s": DT,
                 "cycles": CYCLES, "averaged_from_s": T_RESET or 0.0, "averaged_to_s": T_END,
                 "periodic_change": PERIODIC_CHANGE, "n_faces": int(m.sum()), "area_m2": float(a.sum()),
                 "tawss_mean_pa": float(np.average(tw, weights=a)), "tawss_max_pa": float(tw.max()),
                 **{f"low_tawss_frac_lt_{t:g}": float(a[tw < t].sum() / a.sum()) for t in cfg["low_wss_thresholds"]},
                 "osi_mean": float(np.average(os_, weights=a)), "osi_max": float(os_.max()),
                 f"high_osi_frac_gt_{OSI_THR:g}": float(a[os_ > OSI_THR].sum() / a.sum()),
                 "rrt_mean_finite_inv_pa": float(np.average(rrt[fin], weights=a[fin])) if fin.any() else float("nan"),
                 "rrt_p90_inv_pa": float(rrt[order][min(int(np.searchsorted(cum, 0.9 * cum[-1])), len(cum) - 1)]),
                 "rrt_nonfinite_frac": float(a[~fin].sum() / a.sum()),
                 "region": "25 mm artery segment wall", "case_stem": f"{DESIGN}_{STATE}_nostrut_pulsatile",
                 "lineage": LINEAGE, "udf": ";".join(f.name for f in INLET_C + OSI_C)}
    old = pd.read_csv(OSI_RESULTS) if OSI_RESULTS.exists() else pd.DataFrame()
    if len(old):
        old = old[~((old["design"].astype(str) == DESIGN) & (old["state"].astype(str) == STATE))]
    pd.concat([old, pd.DataFrame([TAWSS_OSI])], ignore_index=True).to_csv(OSI_RESULTS, index=False)
    WSS_SEG = {"wss_mean_pa": TAWSS_OSI["tawss_mean_pa"], "wss_max_pa": TAWSS_OSI["tawss_max_pa"],
               **{f"low_wss_frac_lt_{t:g}": TAWSS_OSI[f"low_tawss_frac_lt_{t:g}"] for t in cfg["low_wss_thresholds"]}}
    print(f"TAWSS / OSI -> {OSI_RESULTS}")
    print(pd.Series(TAWSS_OSI).to_string())

## 28. Write case/data, exports and summary

In [ ]:
s.file.write(file_type="case-data", file_name=str((CASE_DIR / f"{TAG}.cas.h5").resolve()))

base_q = ["pressure", "x-velocity", "y-velocity", "z-velocity", "velocity-magnitude",
          "strain-rate-mag", "viscosity-lam"]
turb_q = ["turb-kinetic-energy", "specific-diss-rate", "viscosity-turb"] if cfg["viscous_model"] == "sst" else []
wall_q = ["pressure", "wall-shear", "x-wall-shear", "y-wall-shear", "z-wall-shear",
          "strain-rate-mag", "viscosity-lam"]
jobs = [("wall", ZONE_WALL, wall_q)]
if ZONE_SYM:
    jobs.append(("symmetry", ZONE_SYM, base_q + turb_q))
EXPORTS = {}
for label, surfaces, quantities in jobs:
    fn = (CASE_DIR / f"{TAG}_{label}.csv").resolve()
    s.file.export.ascii(file_name=str(fn), surface_name_list=list(surfaces), delimiter="comma",
                        quantities=quantities, location="node")     # 26.1 argument names (Task 0.3, 2 Oct)
    EXPORTS[label] = str(fn)

RECORD = dict(
    tag=TAG, created=datetime.datetime.now().isoformat(timespec="seconds"),
    fluent=solver.get_fluent_version(), procs=PROCS, mesh=str(MESH_PATH.resolve()), setup_csv=SETUP_CSV,
    inlet_u=INLET_U, inlet_profile=cfg["inlet_profile"], inlet_expression=INLET_EXPR,
    viscous_model=cfg["viscous_model"], **FLOW,
    A_inlet=A_IN, A_inlet_ratio=A_IN / FLOW["A_inlet_nominal_m2"], R_inlet=R_IN,
    mdot_target=MDOT_TARGET, mdot_init=MDOT_INIT,
    iterations=N_IT, extra_batches=BATCHES, final_residuals=RES_LAST, residuals_met=RES_OK,
    dp_flat=DP_FLAT, dp_span_last_window_pa=DP_SPAN, converged=CONVERGED,
    monitor_spread=SPREAD, spread_ok=SPREAD_OK, mass_imbalance=IMBALANCE, mass_ok=MASS_OK,
    dp_full_pa=VALS["p_inlet"], segment_z_m=[Z0, Z1], p_seg_up_pa=SEG_P["p_seg_up"], p_seg_dn_pa=SEG_P["p_seg_dn"],
    dp_segment25_pa=DP_SEG, wss_wall_areaavg_pa=VALS["wss_wall_areaavg"], wss_wall_max_pa=VALS["wss_wall_max"],
    wss_segment=WSS_SEG, wss_whole_artery_wall=WSS_WALL, wss_strut_walls=WSS_STRUT, vmax_mps=VALS["vmax"],
    tawss_osi=(TAWSS_OSI if OSI_ON else "NOT MEASURED (no UDF in dropin/tawss_osi)"),
    transient=(dict(udf=[f.name for f in INLET_C], period_s=PERIOD, time_step_s=DT, cycles=CYCLES,
                    iterations_per_step=NITER, dp_last_cycle_mean_pa=DP_CYCLE, periodic_change=PERIODIC_CHANGE)
               if TRANSIENT else None),
    design=DESIGN, state=STATE, lineage=LINEAGE, lumen=LUMEN_REC, mesh_report=MESH_REPORT,
    status=STATUS, wall_time_s=WALL_TIME, exports=EXPORTS,
)
(CASE_DIR / f"{TAG}_summary.json").write_text(json.dumps(RECORD, indent=2,
    default=lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else str(o)))   # FluentVersion -> text

summary_csv = out / "run_summary.csv"
cols = ["tag", "inlet_u", "viscous_model", "iterations", "status", "converged", "mass_imbalance",
        "dp_full_pa", "dp_segment25_pa", "vmax_mps", "A_inlet_ratio", "wall_time_s"]
row = pd.DataFrame([{**{c: RECORD[c] for c in cols},
                     "wss_seg_mean_pa": WSS_SEG["wss_mean_pa"], "wss_seg_max_pa": WSS_SEG["wss_max_pa"],
                     **{f"seg_{k}": v for k, v in WSS_SEG.items() if k.startswith("low_wss")}}])
row.to_csv(summary_csv, mode="a", header=not summary_csv.exists(), index=False)
print("written:", CASE_DIR)

## 29. The row for Vishesh's `cfd_stage.csv`
This run's row (his 33 columns, his `classify_case`) is merged into `<CAMPAIGN_OUT>/<DESIGN>/<STATE>_struts-no/cfd_stage.csv` (his `merge_rows`: the other rows stay); `cfd_result.json` goes in the run folder. `run_cfd_stage.py` writes `STAGE_DONE` when every inflow has run, and a FAILED / SKIPPED row for a run that stopped. For a pulsatile run: one row, `U_mps` = the waveform mean, Δp = last-cycle means, WSS columns = TAWSS over the segment (blank without the TAWSS / OSI UDF), a note in `error`.

In [ ]:
import csv
w = WSS_SEG if (OSI_ON or not TRANSIENT) else {}    # transient without the OSI UDF: no cycle-mean WSS
ROW = {"design": DESIGN, "state": STATE, "struts": "no", "role": "production", "flow_model": "production",
       "viscosity": cfg["viscosity_model"], "turbulence": cfg["viscous_model"], "U_mps": INLET_U,
       "class": PC.classify_case({"status": "OK", "converged": CONVERGED}), "converged": CONVERGED,
       "iterations": N_IT, "cells": MESH_REPORT.get("cell_count"),
       "min_orth_quality_mesher": MESH_REPORT.get("min_orthogonal_quality"),
       "stats_source": MESH_REPORT.get("stats_source"),
       "mesh_min_mm": MESH_CTRL.min_size_mm, "mesh_max_mm": MESH_CTRL.max_size_mm,
       "dp_full_pa": DP_CYCLE if TRANSIENT else VALS["p_inlet"], "dp_segment25_pa": DP_SEG,
       "wss_max_pa": w.get("wss_max_pa"), "wss_mean_pa": w.get("wss_mean_pa"),
       "low_wss_frac_lt_0.5": w.get("low_wss_frac_lt_0.5"), "low_wss_frac_lt_0.4": w.get("low_wss_frac_lt_0.4"),
       "low_wss_frac_lt_1": w.get("low_wss_frac_lt_1"),
       "Re_carreau_wall": FLOW["Re_carreau_wall"], "Re_mu_inf": FLOW["Re_mu_inf"],
       "mld_cfd_mm": 2 * GEOM["throat_radius_mm"],
       "ds_cfd_pct": 100 * (1 - GEOM["throat_radius_mm"] / (cfg["R0"] * 1e3)),
       "recirc_length_mm": None, "walltime_s": WALL_TIME, "fluent_processes": PROCS,
       "case_stem": (f"{DESIGN}_{STATE}_nostrut_pulsatile" if TRANSIENT
                     else f"{DESIGN}_{STATE}_nostrut_production_U{INLET_U:.2f}".replace(".", "p")),
       "lineage": LINEAGE,
       "error": (f"pulsatile: last-cycle means, U = waveform mean; WSS = TAWSS" if OSI_ON else
                 "pulsatile: last-cycle means, U = waveform mean; WSS blank (no TAWSS / OSI UDF)") if TRANSIENT else None}
order = PIPE_YAML["stages"]["blocked" if STATE == "blocked" else "stented_no_strut"]
STAGE_CSV = STAGE_DIR / "cfd_stage.csv"
rows = PC.merge_rows(STAGE_CSV, [ROW], order)
with STAGE_CSV.open("w", newline="", encoding="utf-8") as fh:
    wr = csv.DictWriter(fh, fieldnames=PC.CSV_COLS, lineterminator="\n")
    wr.writeheader()
    wr.writerows(rows)
(CASE_DIR / "cfd_result.json").write_text(json.dumps(
    {**RECORD, "class": ROW["class"], "campaign_row": ROW, "notebook": "artery_pyfluent_final.ipynb"},
    indent=2, default=lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else str(o)))
print(f"{ROW['case_stem']}: {ROW['class']}  dp {ROW['dp_full_pa']:.3f} Pa, segment {ROW['dp_segment25_pa']:.3f} Pa, "
      f"WSS mean {ROW['wss_mean_pa'] if ROW['wss_mean_pa'] is None else round(ROW['wss_mean_pa'], 4)} Pa -> {STAGE_CSV}")
pd.DataFrame([ROW]).T

## 30. Plots
Monitors should be flat at the end; the Carreau points should sit on the curve (quarter meshes); WSS along the vessel.

In [ ]:
if MON is not None:
    names = HDR[1:MON.shape[1]]
    fig, axs = plt.subplots(1, len(names), figsize=(2.9 * len(names), 2.5))
    for ax, j, nm in zip(np.atleast_1d(axs), range(1, MON.shape[1]), names):
        ax.plot(MON[:, 0], MON[:, j], lw=1)
        ax.set_title(nm, fontsize=8); ax.set_xlabel("iteration", fontsize=8); ax.tick_params(labelsize=7)
    fig.suptitle(f"{TAG}: {STATUS}", fontsize=9); plt.tight_layout(); plt.show()

fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
if "symmetry" in EXPORTS and cfg["viscosity_model"] == "carreau":
    d = pd.read_csv(EXPORTS["symmetry"]); d.columns = [c.strip() for c in d.columns]
    g = d["strain-rate-mag"].to_numpy()
    gg = np.logspace(np.log10(max(g[g > 0].min(), 1e-3)), np.log10(g.max()), 200)
    axs[0].loglog(g, d["viscosity-lam"], ".", ms=2, alpha=0.4, label="Fluent (symmetry nodes)")
    axs[0].loglog(gg, carreau(gg), "k-", lw=1.2, label="Carreau, CSV constants")
    axs[0].set_xlabel("strain rate [1/s]"); axs[0].set_ylabel("viscosity [Pa s]"); axs[0].legend(fontsize=8)
d = pd.read_csv(EXPORTS["wall"]); d.columns = [c.strip() for c in d.columns]
axs[1].plot(d["z-coordinate"] * 1e3, d["wall-shear"], ".", ms=2, alpha=0.4)
axs[1].set_xlabel("axial position z [mm]"); axs[1].set_ylabel("WSS [Pa]")
fig.suptitle(TAG, fontsize=9); plt.tight_layout(); plt.show()

## 31. Close Fluent

In [ ]:
solver.exit()

## Output layout
```
<CAMPAIGN_OUT>/<design>/<state>_struts-no/
  cfd_stage.csv                    Vishesh's columns, one row per inflow
  STAGE_DONE                       written by run_cfd_stage.py when every inflow has run
  geometry/                        build_lumen output: lumen STEP + STL, geometry.json
  meshes/                          the mesh (shared by the inflows) and its mesh report
  run_U0p26.ipynb, run_U0p10.ipynb executed copy of this notebook per inflow (run_cfd_stage.py)
  run_summary.csv                  one row per run (appended)
  <mesh>_laminar_U0p26/            one folder per run
      case_setup_used.csv          exact CSV used
      *.cas.h5 / *.dat.h5          Fluent case and data
      *_monitors.out               monitor history
      *_faces_<wall>.csv           wall faces: WSS and components, face area, face centre
      *_wall.csv, *_symmetry.csv   node exports for plots
      *_summary.json, cfd_result.json   full record of the run
```